# Tree normalising flows for phylogenetic VI

A **tree normalising flow** puts a flexible, tree-structured distribution on the
unconstrained node-height coordinates that sit underneath treeflow's node-height
ratio transform. Each layer is a *traversal sandwich*:

1. a **postorder** (tip-to-root) affine map -- each node reads its children's outputs;
2. a **learnable elementwise nonlinearity** -- shared across the tree with
   per-node affine conditioning (no traversal). It is pluggable: a monotone
   rational-quadratic spline (the default), `sinh_arcsinh`, or none at all
   (`"affine"`), which reduces the flow to a Gaussian with tree-structured
   covariance;
3. a **preorder** (root-to-tip) affine map -- each node reads its parent's output.

Up-then-down is the collect/distribute schedule of belief propagation, and it is
what makes a *single* layer couple every pair of nodes: node `i` ends up
depending on the descendants of its ancestors, i.e. on everything. The flow is
optionally **conditional** -- on the other model parameters (clock rate,
population size, substitution parameters) and on per-branch parameters -- so the
tree block can depend on the rest of the model.

Both affine maps are triangular in their own traversal order, so the
log-det-Jacobian is `sum_i log scale[i]` and the inverse is a gather; the forward
sweeps run either through the pure-TensorFlow traversal primitives or through
the native C++ ops.

This notebook compares the flow with the **root full-rank** approximation
already in treeflow (`get_fixed_topology_root_full_rank_approximation`: one
full-rank Gaussian block over the non-tree parameters plus the root height, and
mean-field over the remaining ratios), in two studies:

* **Study 1 -- heights only.** Every non-tree parameter is fixed, so the target
  is the posterior over node heights alone. This isolates what the flow buys
  *within* the tree.
* **Study 2 -- full model.** An inverse autoregressive flow on the other
  parameters and the tree flow on the node heights, conditioned on those
  parameters.

Both studies reuse treeflow's existing model definitions (`PhyloModel`, the HKY +
discretised-gamma likelihood, the constant coalescent) and its native likelihood.

## Setup

In [ ]:
import numpy as np
import yaml
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
from tqdm import tqdm

tf.get_logger().setLevel("ERROR")
tfd = tfp.distributions

from treeflow import (
    Alignment,
    AlignmentFormat,
    parse_newick,
    PhyloModel,
    convert_tree_to_tensor,
)
from treeflow.model.phylo_model import phylo_model_to_joint_distribution
from treeflow.distributions import (
    DiscretizedDistribution,
    DiscreteParameterMixture,
    LeafCTMC,
)
from treeflow.distributions.tree import ConstantCoalescent
from treeflow.evolution.substitution import HKY, get_transition_probabilities_tree
from treeflow.model.approximation import (
    get_fixed_topology_mean_field_approximation,
    get_fixed_topology_full_rank_approximation,
    get_fixed_topology_root_full_rank_approximation,
    get_fixed_topology_tree_flow_approximation,
)
from treeflow.vi import fit_fixed_topology_variational_approximation, RobustOptimizer

In [ ]:
NUM_STEPS = 2_500        # optimisation steps per fit (budget-matched across families)
LEARNING_RATE = 0.01
SAMPLE_SIZE = 4          # Monte Carlo samples per ELBO gradient
ELBO_SAMPLES = 512       # samples for the final (low-noise) ELBO estimate
ELBO_CHUNK = 32          # ELBO samples evaluated per likelihood call
N_SAMPLES = 4_000        # posterior samples drawn per fit for the marginals
SEED = (1, 1)
DTYPE = tf.float64

In [ ]:
alignment = Alignment("../examples/demo-data/YFV.nex", format=AlignmentFormat.NEXUS)
starting_tree = parse_newick("../examples/demo-data/YFV.newick")
starting_tensor_tree = convert_tree_to_tensor(starting_tree)
sequence_tensor = alignment.get_encoded_sequence_tensor(starting_tree.taxon_set)
topologies = dict(tree=starting_tree.topology)
taxon_count = starting_tree.taxon_count
node_count = taxon_count - 1  # internal nodes == the flow's coordinates
print("taxa:", taxon_count, "| sites:", alignment.site_count, "| flow dimension:", node_count)

### Model

The same HKY + discretised-gamma + strict-clock + constant-coalescent model the
other ADVI notebooks in this directory use, so the numbers here are comparable
with `advi_iaf.ipynb`.

In [ ]:
model_string = """
clock:
  strict:
    clock_rate:
      lognormal:
        loc: -2.0
        scale: 2.0
site:
  discrete_gamma:
    category_count: 4
    site_gamma_shape:
      lognormal:
        loc: 0.0
        scale: 1.0
substitution:
  hky:
    kappa:
      lognormal:
        loc: 1.0
        scale: 1.25
    frequencies:
      dirichlet:
        concentration:
        - 2.0
        - 2.0
        - 2.0
        - 2.0
tree:
  coalescent:
    pop_size:
      lognormal:
        loc: 1.0
        scale: 1.5
"""

phylo_model = PhyloModel(yaml.safe_load(model_string))
base_model_dist = phylo_model_to_joint_distribution(
    phylo_model, starting_tensor_tree, alignment
)
prior_dists = {d.name: d for d in base_model_dist._get_single_sample_distributions()}
site_category_count = 4
subst_model = HKY()

In [ ]:
def sequence_distribution(tree, clock_rate, kappa, frequencies, site_gamma_shape):
    """The (native-accelerated) phylogenetic likelihood term."""
    # The trailing axis keeps a batch of samples broadcasting against the
    # per-branch axis: clock_rate is [...] and branch_lengths [..., branch].
    distance_tree = tree.get_unrooted_tree() * clock_rate[..., tf.newaxis]
    site_rate_distribution = DiscretizedDistribution(
        category_count=site_category_count,
        distribution=tfd.Gamma(site_gamma_shape, site_gamma_shape),
    )
    transition_probs_tree = get_transition_probabilities_tree(
        distance_tree,
        subst_model,
        rate_categories=site_rate_distribution.normalised_support,
        frequencies=frequencies,
        kappa=kappa,
    )
    leaf_ctmc = LeafCTMC(
        transition_probs_tree, tf.expand_dims(frequencies, -2), use_native=True
    )
    site_mixture = DiscreteParameterMixture(site_rate_distribution, leaf_ctmc)
    return tfd.Sample(site_mixture, sample_shape=alignment.site_count)


def build_full_model():
    """The joint distribution with every parameter free, pinned on the alignment."""
    def alignment_dist(kappa, frequencies, tree, clock_rate, site_gamma_shape):
        return sequence_distribution(
            tree, clock_rate, kappa, frequencies, site_gamma_shape
        )

    return tfd.JointDistributionNamed(
        dict(
            pop_size=prior_dists["pop_size"],
            kappa=prior_dists["kappa"],
            frequencies=prior_dists["frequencies"],
            site_gamma_shape=prior_dists["site_gamma_shape"],
            clock_rate=prior_dists["clock_rate"],
            tree=lambda pop_size: ConstantCoalescent(
                starting_tensor_tree.taxon_count,
                pop_size,
                starting_tensor_tree.sampling_times,
                tree_name="tree",
            ),
            alignment=alignment_dist,
        )
    ).experimental_pin(alignment=sequence_tensor)

In [ ]:
# Prior medians in constrained space, as a deterministic starting point (the same
# convention as advi_iaf.ipynb), so differences between families are not
# initialisation luck.
INIT_LOC = dict(
    tree=starting_tensor_tree,
    clock_rate=tf.constant(np.exp(-2.0), dtype=DTYPE),
    kappa=tf.constant(np.exp(1.0), dtype=DTYPE),
    pop_size=tf.constant(np.exp(1.0), dtype=DTYPE),
    site_gamma_shape=tf.constant(np.exp(0.0), dtype=DTYPE),
    frequencies=tf.constant([0.25, 0.25, 0.25, 0.25], dtype=DTYPE),
)


def fit(model, approx_fn, approx_kwargs=None, init_loc=None, num_steps=NUM_STEPS,
        seed=SEED, label=""):
    """Fit one variational family and return the approximation and its loss trace."""
    optimizer = RobustOptimizer(tf.optimizers.Adam(learning_rate=LEARNING_RATE))
    approximation, trace = fit_fixed_topology_variational_approximation(
        model,
        topologies=topologies,
        optimizer=optimizer,
        num_steps=num_steps,
        init_loc=init_loc,
        approx_fn=approx_fn,
        approx_kwargs=approx_kwargs or {},
        sample_size=SAMPLE_SIZE,
        progress_bar=tqdm,
        seed=seed,
    )
    loss = np.asarray(trace.loss)
    print(f"{label}: {len(loss)} steps, final training ELBO ~ {-loss[-100:].mean():.2f}")
    return approximation, loss


def estimate_elbo(approximation, model, num_samples=ELBO_SAMPLES, seed=(42, 42)):
    """Low-noise ELBO estimate (and its Monte Carlo standard error).

    The ELBO is the like-for-like comparison here: every family approximates the
    same target, so a higher ELBO means a smaller KL to the true posterior.
    Evaluated in chunks: the likelihood's memory grows with the sample count.
    """
    seeds = tfp.random.split_seed(seed, n=num_samples // ELBO_CHUNK)
    elbo_samples = []
    for chunk_seed in seeds:
        sample, log_q = approximation.experimental_sample_and_log_prob(
            ELBO_CHUNK, seed=chunk_seed
        )
        elbo_samples.append(
            (model.unnormalized_log_prob(sample) - log_q).numpy()
        )
    elbo_samples = np.concatenate(elbo_samples)
    elbo_samples = elbo_samples[np.isfinite(elbo_samples)]
    return elbo_samples.mean(), elbo_samples.std() / np.sqrt(elbo_samples.size)

## Study 1 -- node heights only

Every non-tree parameter is held at a point value, so the target is purely the
posterior over the `taxon_count - 1` node heights. This is the setting in which
the flow's tree structure is the *only* thing being tested.

Worth being explicit about one thing: with no free non-tree parameters, the
**root full-rank approximation degenerates to mean-field**. Its full-rank block
covers the non-tree parameters *plus the root height*, so with nothing else in
the block it is a one-dimensional Gaussian and the remaining ratios stay
independent. That is exactly the point of this study -- the root full-rank family
buys correlation between the root and the other parameters, and nothing at all
*within* the tree. The whole-tree full-rank Gaussian is included as the reference
for what a Gaussian with unrestricted covariance can do here.

The parameters are fixed at the posterior means of a mean-field fit to the full
model (rather than at prior medians), so the heights are being inferred at
realistic parameter values. That fit doubles as study 2's baseline.

In [ ]:
full_model = build_full_model()

mean_field_full, mean_field_full_loss = fit(
    full_model,
    get_fixed_topology_mean_field_approximation,
    init_loc=INIT_LOC,
    label="full model / mean-field",
)

In [ ]:
# Posterior means of the non-tree parameters, used as study 1's fixed values.
_mf_sample = mean_field_full.sample(N_SAMPLES, seed=(7, 7))
FIXED_PARAMETERS = {
    name: tf.reduce_mean(_mf_sample[name], axis=0)
    for name in ("clock_rate", "kappa", "frequencies", "site_gamma_shape", "pop_size")
}
pd.Series(
    {
        name: np.round(value.numpy(), 4).tolist()
        for name, value in FIXED_PARAMETERS.items()
    }
).to_frame("fixed value")

In [ ]:
def build_heights_only_model():
    """The same model with every non-tree parameter fixed: only `tree` is free."""
    def alignment_dist(tree):
        return sequence_distribution(
            tree,
            FIXED_PARAMETERS["clock_rate"],
            FIXED_PARAMETERS["kappa"],
            FIXED_PARAMETERS["frequencies"],
            FIXED_PARAMETERS["site_gamma_shape"],
        )

    return tfd.JointDistributionNamed(
        dict(
            tree=ConstantCoalescent(
                starting_tensor_tree.taxon_count,
                FIXED_PARAMETERS["pop_size"],
                starting_tensor_tree.sampling_times,
                tree_name="tree",
            ),
            alignment=alignment_dist,
        )
    ).experimental_pin(alignment=sequence_tensor)


heights_model = build_heights_only_model()

### The families being compared, and why these ablations

Alongside the Gaussian baselines, study 1 varies two things about the flow:

* **The traversal order.** `postorder_first` (up then down) couples every pair of
  node heights with one layer; `preorder_first` couples only ancestor-descendant
  pairs until a second layer is added. The `preorder first` and `2 layers` runs
  are there to see whether that structural difference shows up in the ELBO.
* **The nonlinearity.** `affine only` removes it entirely, leaving the two
  triangular affine maps -- a Gaussian with tree-structured covariance in
  `O(internal_node)` parameters rather than the `O(internal_node^2)` of a full
  covariance. `sinh-arcsinh` swaps the spline's bounded, flexible-in-the-bulk
  nonlinearity for an unbounded one that reshapes the tails instead.

The tails are worth a note. The flow's coordinates pass through a sigmoid (the
non-root ratios) or an exponential (the root height) before the ratio transform,
so a spline that is the identity outside `[-bounds, bounds]` leaves logit-normal
tails on the ratios -- exactly the tails the existing mean-field and full-rank
families have, so nothing is lost relative to them, and the linear tails keep the
inverse and the log-det-Jacobian well conditioned. What the spline changes is the
bulk, and each node's learnable affine pre-conditioning is what positions that
bulk inside the spline's window. `sinh-arcsinh` is the option to reach for when
the tails themselves need to change; comparing the two here is the test of
whether that matters for this posterior.

In [ ]:
study1_families = {
    "mean field": (get_fixed_topology_mean_field_approximation, {}),
    "root full rank": (get_fixed_topology_root_full_rank_approximation, {}),
    "full rank": (get_fixed_topology_full_rank_approximation, {}),
    "tree flow": (
        get_fixed_topology_tree_flow_approximation,
        dict(num_layers=1, order="postorder_first"),
    ),
    "tree flow (preorder first)": (
        get_fixed_topology_tree_flow_approximation,
        dict(num_layers=1, order="preorder_first"),
    ),
    "tree flow (2 layers)": (
        get_fixed_topology_tree_flow_approximation,
        dict(num_layers=2, order="postorder_first"),
    ),
    # Ablation: the same sandwich with no nonlinearity at all, i.e. a Gaussian
    # with tree-structured covariance in O(internal_node) parameters. The gap
    # between this and "tree flow" is what the spline buys; the gap between this
    # and "full rank" is what the O(N^2) Gaussian buys over the structured one.
    "tree flow (affine only)": (
        get_fixed_topology_tree_flow_approximation,
        dict(num_layers=1, nonlinearity="affine"),
    ),
    # The unbounded nonlinearity: reshapes the tails rather than the bulk.
    "tree flow (sinh-arcsinh)": (
        get_fixed_topology_tree_flow_approximation,
        dict(num_layers=1, nonlinearity="sinh_arcsinh"),
    ),
}

study1 = {}
for label, (approx_fn, approx_kwargs) in study1_families.items():
    approximation, loss = fit(
        heights_model,
        approx_fn,
        approx_kwargs,
        init_loc=dict(tree=starting_tensor_tree),
        label=f"heights only / {label}",
    )
    study1[label] = dict(approximation=approximation, loss=loss)

In [ ]:
for label, result in study1.items():
    elbo, standard_error = estimate_elbo(result["approximation"], heights_model)
    result["elbo"], result["elbo_se"] = elbo, standard_error

study1_table = pd.DataFrame(
    {
        label: {
            "ELBO": result["elbo"],
            "MC std. error": result["elbo_se"],
            "vs root full rank": result["elbo"] - study1["root full rank"]["elbo"],
        }
        for label, result in study1.items()
    }
).T
study1_table.round(2)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))

window = 50
for label, result in study1.items():
    smoothed = np.convolve(
        -result["loss"], np.ones(window) / window, mode="valid"
    )
    axs[0].plot(smoothed, label=label, lw=1)
axs[0].set_xlabel("step")
axs[0].set_ylabel(f"ELBO ({window}-step moving average)")
axs[0].set_title("Study 1: optimisation traces")
axs[0].legend(fontsize=8)

lower = min(result["elbo"] for result in study1.values())
axs[1].bar(
    range(len(study1)),
    [result["elbo"] - lower for result in study1.values()],
    color="steelblue",
)
axs[1].set_xticks(range(len(study1)))
axs[1].set_xticklabels(study1.keys(), rotation=30, ha="right", fontsize=8)
axs[1].set_ylabel(f"ELBO - {lower:.0f}")
axs[1].set_title("Study 1: final ELBO (2048 samples)")
plt.tight_layout()

### What the flow does to the tree posterior

Two things a mean-field approximation over node heights cannot represent, and
that the flow can: **correlation between node heights** (a clade's age and its
parent's age move together) and **non-Gaussian marginals** (node heights are
squeezed against their descendants' heights, which skews them).

In [ ]:
study1_samples = {
    label: result["approximation"].sample(N_SAMPLES, seed=(11, 11))
    for label, result in study1.items()
}
height_samples = {
    label: sample["tree"].node_heights.numpy()
    for label, sample in study1_samples.items()
}

compare = ["mean field", "full rank", "tree flow (affine only)", "tree flow"]
fig, axs = plt.subplots(1, len(compare), figsize=(5 * len(compare), 4))
for ax, label in zip(axs, compare):
    correlations = np.corrcoef(height_samples[label].T)
    image = ax.imshow(correlations, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_title(f"node-height correlation\n{label}")
    ax.set_xlabel("internal node")
    ax.set_ylabel("internal node")
    plt.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()

off_diagonal = ~np.eye(node_count, dtype=bool)
pd.Series(
    {
        label: np.abs(np.corrcoef(samples.T)[off_diagonal]).mean()
        for label, samples in height_samples.items()
    },
    name="mean |correlation| between node heights",
).to_frame().round(3)

In [ ]:
def skewness(samples, axis=0):
    centred = samples - samples.mean(axis=axis, keepdims=True)
    return (centred ** 3).mean(axis=axis) / centred.std(axis=axis) ** 3


# Marginal shape: how far each family's node-height marginals are from Gaussian.
skew_table = pd.DataFrame(
    {
        label: {
            "mean |skew| over nodes": np.abs(skewness(samples)).mean(),
            "root height mean": samples[:, -1].mean(),
            "root height sd": samples[:, -1].std(),
        }
        for label, samples in height_samples.items()
    }
).T
skew_table.round(4)

In [ ]:
# The parent-child pair with the strongest dependence under the flow, drawn for
# each family: this is the geometry mean-field factorises away.
parent_indices = (
    starting_tree.topology.parent_indices[taxon_count:] - taxon_count
)
flow_correlations = np.corrcoef(height_samples["tree flow"].T)
pairs = [(child, int(parent)) for child, parent in enumerate(parent_indices)]
child, parent = max(pairs, key=lambda pair: abs(flow_correlations[pair]))

fig, axs = plt.subplots(1, len(compare), figsize=(4.5 * len(compare), 4))
for ax, label in zip(axs, compare):
    samples = height_samples[label]
    ax.hist2d(samples[:, child], samples[:, parent], bins=60, cmap="Blues")
    ax.set_xlabel(f"height of node {child}")
    ax.set_ylabel(f"height of parent node {parent}")
    ax.set_title(
        f"{label}\ncorr = {np.corrcoef(samples[:, child], samples[:, parent])[0, 1]:.2f}"
    )
plt.tight_layout()

### Study 1 findings

*(Filled in from the run above.)*

## Study 2 -- full model inference

Now everything is free. The tree flow approximation is built as

* an **inverse autoregressive flow** over the non-tree parameters
  (`parameter_approximation="iaf"`), and
* the **tree flow** over the node heights, *conditioned* on those parameters'
  unconstrained values.

The coupling is one-way, so the Jacobian is block-triangular and the density
stays exact. This is compared with the root full-rank approximation, whose whole
purpose is the correlation between the tree's root height and the other
parameters -- the clock rate x root height ridge that `advi_iaf.ipynb`
diagnoses -- and with plain mean field.

In [ ]:
study2_families = {
    "mean field": (get_fixed_topology_mean_field_approximation, {}),
    "root full rank": (get_fixed_topology_root_full_rank_approximation, {}),
    "tree flow + IAF": (
        get_fixed_topology_tree_flow_approximation,
        dict(num_layers=1, parameter_approximation="iaf"),
    ),
    "tree flow (unconditional)": (
        get_fixed_topology_tree_flow_approximation,
        dict(num_layers=1, auxiliary_vars=()),
    ),
}

study2 = {"mean field": dict(approximation=mean_field_full, loss=mean_field_full_loss)}
for label, (approx_fn, approx_kwargs) in study2_families.items():
    if label in study2:
        continue  # the mean-field fit was run above as study 1's warm-up
    approximation, loss = fit(
        full_model,
        approx_fn,
        approx_kwargs,
        init_loc=INIT_LOC,
        label=f"full model / {label}",
    )
    study2[label] = dict(approximation=approximation, loss=loss)

In [ ]:
for label, result in study2.items():
    elbo, standard_error = estimate_elbo(result["approximation"], full_model)
    result["elbo"], result["elbo_se"] = elbo, standard_error

study2_table = pd.DataFrame(
    {
        label: {
            "ELBO": result["elbo"],
            "MC std. error": result["elbo_se"],
            "vs root full rank": result["elbo"] - study2["root full rank"]["elbo"],
        }
        for label, result in study2.items()
    }
).T
study2_table.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, result in study2.items():
    smoothed = np.convolve(-result["loss"], np.ones(window) / window, mode="valid")
    ax.plot(smoothed, label=label, lw=1)
ax.set_xlabel("step")
ax.set_ylabel(f"ELBO ({window}-step moving average)")
ax.set_title("Study 2: optimisation traces")
ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
study2_samples = {
    label: result["approximation"].sample(N_SAMPLES, seed=(13, 13))
    for label, result in study2.items()
}

POSTERIOR_PARAMS = ["pop_size", "kappa", "site_gamma_shape", "clock_rate"]


def marginal(sample, name):
    if name == "root_height":
        return sample["tree"].node_heights[:, -1].numpy()
    return sample[name].numpy()


fig, axs = plt.subplots(1, len(POSTERIOR_PARAMS) + 1, figsize=(4 * (len(POSTERIOR_PARAMS) + 1), 3.5))
for ax, name in zip(axs, POSTERIOR_PARAMS + ["root_height"]):
    for label, sample in study2_samples.items():
        ax.hist(marginal(sample, name), bins=60, histtype="step", density=True, label=label)
    ax.set_title(name)
axs[0].legend(fontsize=7)
plt.tight_layout()

In [ ]:
# The clock rate x root height ridge: the dependence the root full-rank family
# was built for, and the one the conditional flow models through its auxiliary
# input.
fig, axs = plt.subplots(1, len(study2_samples), figsize=(4.5 * len(study2_samples), 4))
for ax, (label, sample) in zip(axs, study2_samples.items()):
    clock = np.log(marginal(sample, "clock_rate"))
    height = np.log(marginal(sample, "root_height"))
    ax.hist2d(clock, height, bins=60, cmap="Blues")
    ax.set_xlabel("log clock_rate")
    ax.set_ylabel("log root_height")
    ax.set_title(f"{label}\ncorr = {np.corrcoef(clock, height)[0, 1]:.2f}")
plt.tight_layout()

In [ ]:
summary = pd.DataFrame(
    {
        label: {
            "clock-root corr (log)": np.corrcoef(
                np.log(marginal(sample, "clock_rate")),
                np.log(marginal(sample, "root_height")),
            )[0, 1],
            "mean |height correlation|": np.abs(
                np.corrcoef(sample["tree"].node_heights.numpy().T)[off_diagonal]
            ).mean(),
            "ELBO": study2[label]["elbo"],
        }
        for label, sample in study2_samples.items()
    }
).T
summary.round(3)

### Study 2 findings

*(Filled in from the run above.)*

## Notes on cost

The flow adds `(7 + num_children) * internal_node` free parameters per layer plus
a shared spline, and two extra tree traversals per ELBO evaluation. Those
traversals run through the native C++ ops when they are built
(`python -m treeflow.acceleration.native.build`, or `use_native=False` to force
the pure-TensorFlow path); their log-det-Jacobian costs nothing beyond a sum of
logs, and their inverses are gathers rather than traversals.